# Lab 4: Prompt design and measuring it
**Module 2.2: Prompt design.**  **Time:** 30 minutes.

**Goal:** improve a prompt and **prove** it improved, using 20 labelled support tickets.
Prompts are code: version them and test them.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
import pandas as pd

In [ ]:
tickets = pd.read_csv("data/tickets.csv")
print(len(tickets), "tickets")
tickets.head()

## Prompt versions
`v1` is what most people write first. `v2` adds a role, clear category definitions, an output format and
examples (few-shot). Both are stored with a version id, as you would in a prompt registry.

In [ ]:
CATEGORIES = ["billing", "delivery", "returns", "account", "technical"]

PROMPTS = {
    "v1": {
        "system": None,
        "template": "What category is this customer ticket? {ticket}",
    },
    "v2": {
        "system": """You classify customer support tickets for an online retailer.
Categories:
- billing: payments, charges, refunds not received, invoices, EMI
- delivery: parcel late, missing, wrong item, courier behaviour
- returns: returning or exchanging a product, faulty or expired product
- account: login, password, OTP, account security, profile changes, data deletion
- technical: app or website errors, slowness, promo code not working
Priority: high if money was lost or account security is at risk, low for simple questions, else medium.
Reply with JSON only: {"category": "...", "priority": "..."}""",
        "template": """Examples:
Ticket: "Card charged but no order confirmation" -> {{"category": "billing", "priority": "high"}}
Ticket: "Can I swap this jacket for a bigger size?" -> {{"category": "returns", "priority": "low"}}
Ticket: "Login page shows error 500" -> {{"category": "technical", "priority": "medium"}}

Ticket: "{ticket}" ->""",
    },
}

In [ ]:
def classify(ticket: str, version: str) -> str:
    p = PROMPTS[version]
    out = lk.chat(p["template"].format(ticket=ticket), system=p["system"], model="small",
                  max_tokens=60, tag=f"prompt:{version}")
    try:
        cat = str(lk.parse_json(out).get("category", "")).lower()
    except Exception:  # v1 returns prose, so we look for a category word in it
        cat = next((c for c in CATEGORIES if c in out.lower()), "unparsed")
    return cat

def accuracy(version: str):
    preds = [classify(t, version) for t in tickets.ticket]
    df = tickets.assign(predicted=preds, correct=[p == c for p, c in zip(preds, tickets.category)])
    print(f"{version}: accuracy {df.correct.mean():.0%}")
    return df

r1 = accuracy("v1")

In [ ]:
r2 = accuracy("v2")

## Which tickets did each version get wrong?

In [ ]:
compare = tickets[["ticket", "category"]].assign(v1=r1.predicted, v2=r2.predicted)
compare[(compare.v1 != compare.category) | (compare.v2 != compare.category)]

## Try this 1: do the examples matter? (4 minutes)
The next cell creates **v3**: the same as v2 but **without the three examples**. Run it and compare the accuracy with v2.

In [ ]:
PROMPTS["v3"] = {"system": PROMPTS["v2"]["system"], "template": 'Ticket: "{ticket}" ->'}
r3 = accuracy("v3")

**Question:** did removing the examples change the accuracy? Would you keep them, given they add tokens to every call?

## Try this 2: is the model wrong, or the label? (3 minutes)
The next cell lists the tickets where v2 disagrees with the label. Read each one.

In [ ]:
compare[compare.v2 != compare.category][["ticket", "category", "v2"]]

**Question:** for each ticket listed, is the model wrong, or is the label debatable? (For example, "Refund for returned
TV not received" is labelled *billing*; *returns* is also defensible.) Real test sets always contain debatable labels,
so teams need written labelling rules.

## Discussion (end of lab)
Who owns prompts in your organisation? Where would `PROMPTS` live in production (code repository, configuration
service, prompt registry), and who approves a change?